# Day 7: Incremental Processing & CDC Upsert Engine

This notebook simulates idempotent UPSERT/DELETE logic on a simulated stateful table handling out-of-order CDC updates.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day07_Incremental_Upsert") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session with Iceberg initialized.")

In [ ]:
# Setup Target Table
spark.sql("CREATE DATABASE IF NOT EXISTS local.silver")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.silver.patients (
    patient_id STRING,
    status STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Insert initial state
spark.sql("""
INSERT INTO local.silver.patients VALUES 
('P1', 'Active', timestamp '2023-10-01 10:00:00'),
('P2', 'Active', timestamp '2023-10-01 10:00:00')
""")
print("Initial State:")
spark.sql("SELECT * FROM local.silver.patients ORDER BY patient_id").show()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

# Incoming CDC updates (notice out of order for P1)
cdc_data = [
    ("P1", "Discharged", "2023-10-02 12:00:00"), # Newer
    ("P1", "Admitted", "2023-10-02 08:00:00"),   # Older but arrived later
    ("P3", "Active", "2023-10-02 09:00:00")      # New record
]

updates_df = spark.createDataFrame(cdc_data, ["patient_id", "status", "updated_at"])
updates_df = updates_df.withColumn("updated_at", col("updated_at").cast("timestamp"))

# Deduplicate within the batch
window_spec = Window.partitionBy("patient_id").orderBy(desc("updated_at"))
latest_updates_df = updates_df.withColumn("rn", row_number().over(window_spec)) \
                              .filter(col("rn") == 1) \
                              .drop("rn")

latest_updates_df.createOrReplaceTempView("latest_updates")

# Idempotent Merge handling out-of-order relative to target
spark.sql("""
MERGE INTO local.silver.patients t
USING latest_updates s
ON t.patient_id = s.patient_id
WHEN MATCHED AND t.updated_at < s.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("State after Incremental Upsert:")
spark.sql("SELECT * FROM local.silver.patients ORDER BY patient_id").show()